# EduCrew Studio: free AI episode maker
Every step uses free, open-source models running on Kaggle's free GPU, so there are no per-video charges.

**Before you start (one time):**
1. In Kaggle, open **Settings** (right panel) and set **Accelerator** to **GPU T4 x2**. Turn **Internet** on.
2. Upload the `educrew-studio` folder (zipped) as a Kaggle **Dataset** and attach it with **Add Input**.
3. Create a free Hugging Face account. Accept the terms on https://huggingface.co/ai4bharat/indic-parler-tts, create a *read* token, and add it under **Add-ons → Secrets** with the name `HF_TOKEN`.
4. Put your logo in the project's `logo/` folder as `educrew_logo.png`.

Run the cells from top to bottom. Video clips are slow on free GPUs, so you can make a few shots per session.
Before closing, run **Save progress** at the bottom.

In [ ]:
import glob, os, shutil
EPISODE = "episodes/episode01.json"      # change for the next episode
PROJECT = "/kaggle/working/educrew-studio"
if not os.path.exists(PROJECT):
    src = os.path.dirname(glob.glob("/kaggle/input/**/assemble.py", recursive=True)[0])
    shutil.copytree(src, PROJECT)
os.chdir(PROJECT)
# Resume: if you uploaded a saved work.zip from an earlier session, unpack it
for z in glob.glob("/kaggle/input/**/work.zip", recursive=True):
    shutil.unpack_archive(z, PROJECT); print("restored", z)
print(os.listdir())

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))
!pip install -q -U diffusers transformers accelerate ftfy imageio-ffmpeg sentencepiece

## Step 1: keyframes (one photoreal still per shot)
Look at the results. To redo a shot, add `--shots 3 --force --seed 55` to the command.

In [ ]:
!cd tools && python make_keyframes.py ../$EPISODE --work ../work/episode01
from IPython.display import display, Image as Im
for p in sorted(glob.glob("work/episode01/keyframes/*.png")): print(p); display(Im(p, width=220))

## Step 2: video clips (slowest step)
Run a few shots at a time. Finished clips are skipped automatically. Shots without a clip still work: the assembler animates the still image instead.
If you run out of memory, add `--width 480 --height 832`.

In [ ]:
!cd tools && python make_clips.py ../$EPISODE --work ../work/episode01 --shots 1,2,3

## Step 3: voices (narrator, mother, child), exact words from the episode file

In [ ]:
!python -m venv /tmp/tts && /tmp/tts/bin/pip install -q torch soundfile transformers sentencepiece git+https://github.com/huggingface/parler-tts.git
!cd tools && /tmp/tts/bin/python make_voices.py ../$EPISODE --work ../work/episode01
from IPython.display import Audio
for p in sorted(glob.glob("work/episode01/voice/*.wav")): print(p); display(Audio(p))

## Step 4: realistic sound effects (no music)

In [ ]:
if not os.path.exists("/kaggle/working/MMAudio"):
    !git clone -q https://github.com/hkchengrex/MMAudio /kaggle/working/MMAudio
    !pip install -q -e /kaggle/working/MMAudio
!cd tools && python make_sfx.py ../$EPISODE --work ../work/episode01 --mmaudio /kaggle/working/MMAudio

## Step 5: assemble the final 60-second Reel with the logo

In [ ]:
!python assemble.py $EPISODE --work work/episode01
from IPython.display import Video
Video("output/EduCrew_Ep01.mp4", embed=True, width=300)

## Save progress
Download `work.zip` (and the final video from `output/`) from the **Output** panel. Next session, upload `work.zip` as a dataset to continue where you left off.

In [ ]:
shutil.make_archive("/kaggle/working/work", "zip", PROJECT, "work")
print("saved /kaggle/working/work.zip")